In [1]:
# 1 — сжатие(2) 
import os, json, zlib, bz2, lzma, base64
import pandas as pd
try:
    import brotli
except ImportError:
    brotli = None

# === опции ===
CONTENT_TYPE = "notebook"   # "notebook" или "dataframe"
NB_PATH      = "bootstrap_to_excel.ipynb"   # <-- укажите путь к ноутбуку
SOURCE_DF    = None          # если CONTENT_TYPE == "dataframe" — присвойте нужный df
# =============

_ALGO_NAMES = {"Z": "zlib", "B": "bz2", "L": "lzma", "R": "brotli"}

def _compress_best(raw):
    candidates = {"Z": zlib.compress(raw, level=9),
                  "B": bz2.compress(raw, compresslevel=9),
                  "L": lzma.compress(raw, preset=9 | lzma.PRESET_EXTREME)}
    if brotli is not None:
        candidates["R"] = brotli.compress(raw, quality=11)
    tag = min(candidates, key=lambda k: len(candidates[k]))
    return tag, candidates[tag]

if CONTENT_TYPE == "notebook":
    if not NB_PATH:
        raise ValueError("Укажите путь к ноутбуку в переменной NB_PATH")
    if not os.path.isfile(NB_PATH):
        raise FileNotFoundError(f"Файл не найден: {NB_PATH!r}")
    print(f"Ноутбук: {NB_PATH}")
    with open(NB_PATH, "r", encoding="utf-8") as f:
        nb = json.load(f)
    parts = []
    for i, cell in enumerate(nb["cells"], start=1):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        parts.append(f"### CELL {i} [{cell['cell_type']}] ###\n{source}\n")
    raw = "".join(parts).encode("utf-8")
    content_tag = "N"
    print(f"Ячеек: {len(nb['cells'])}")

elif CONTENT_TYPE == "dataframe":
    if SOURCE_DF is None:
        raise ValueError("CONTENT_TYPE == 'dataframe', но SOURCE_DF не задан")
    raw = SOURCE_DF.to_csv(index=False).encode("utf-8")
    content_tag = "D"
    print(f"Датафрейм: {SOURCE_DF.shape[0]} строк, {SOURCE_DF.shape[1]} колонок")

else:
    raise ValueError(f"неизвестный CONTENT_TYPE: {CONTENT_TYPE!r}")

algo_tag, compressed = _compress_best(raw)
_qr_prefix  = content_tag + algo_tag
_qr_payload = base64.b64encode(compressed).decode("ascii")
print(f"Сжатие: {len(raw)} -> {len(compressed)} байт ({_ALGO_NAMES[algo_tag]}), base64: {len(_qr_payload)} симв.")

Ноутбук: bootstrap_to_excel.ipynb
Ячеек: 1
Сжатие: 5018 -> 1902 байт (zlib), base64: 2536 симв.


In [3]:
# 2 — партиции или QR-коды

import io
import qrcode
from IPython.display import display, Image

# === опция ===
OUTPUT_MODE = "text"   # "qr" или "text"
# =============

if OUTPUT_MODE == "qr":
    max_chunk = 2953 - len(f"{_qr_prefix}|001/001|")
elif OUTPUT_MODE == "text":
    max_chunk = 2950
else:
    raise ValueError(f"неизвестный OUTPUT_MODE: {OUTPUT_MODE!r}")

chunks = [_qr_payload[i:i + max_chunk] for i in range(0, len(_qr_payload), max_chunk)]
total = len(chunks)
print(f"Частей: {total}")

for idx, chunk in enumerate(chunks, start=1):
    part = f"{_qr_prefix}|{idx:03d}/{total:03d}|{chunk}"
    print(f"--- часть {idx}/{total} ---")
    if OUTPUT_MODE == "qr":
        qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_L, box_size=8, border=4)
        qr.add_data(part.encode("ascii"))
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        display(Image(data=buf.getvalue()))
    else:
        print(part)

Частей: 1
--- часть 1/1 ---
NZ|001/001|eNqtWF9v28gRf+en2EoISDYMa/ucXGNUD05it8GlueBybR8EgaDIlbUwxSWWK9tJEMBJ2t4Vl/bQwwEtChS4PvTdTWEkudi+r0B+hX6SzuwuKVKS/xSo5D/k7m9+MzszuzNkt9sld7cePCCrpB/xmA5It9u12CTjQpIsTOMwJ/CTxdUYz2iaPTlIrJHgk/rOz+WThObEgLZ5Kj3yKJSSinSbJYlHNhO2k04ojt/hIqbCI49ZTOdoppIlNcsOlUHEk+kkDRKKVJa1/enDz4OHm7/cIj1ibwoWJrb1+P7Pg+37sAQYuntra/vulk3Up0uKt8VpcVR+UbwrTooz4hQ/wN/in8VZ8aE4KV+51sNPm9Igun7vlm1kT4vjywkef3L/US1/7zZ+baO8fFkcly/Kl4YJJOEOmGD8qDglTi5DOc3Jj0CQ79qu9YutzXtbn9Vka9s312/fsi3LiumI0AN0SSBoPk1kHkge0IOIJk41EI88CJcc9+xwGEiay2DIucylCDMQQln/IMkPbM8iF30mVAoWBUk4pEnPLr4rjou35aviQ/karS5fll+RzTs3qpUVR8R5xFme85TU6tzLdIRJNg57K/7KTXdDITudjvo/Wwv5z+G3pDyc1z5TEphV+mKaBmGSqJvciUeuRyA4/yq/hsgdoZ0oDQE4UxrKF6T4XgXvDMbw6j3Jgr0wmdJgzJMJcfAvJJ0QNJKMp73PxZRqa/4IbGezhDiG/+9MXAkYeYK0MH6E4+WfXF8pLL6B4bfFG1jLUfkHmDzGfPihfAWufA22gEWQFr/D6eK0/DPIgqmYWaT8LQy9B8pXfstJbETspsk2SbkkLG14z9ebJt+o4yBCllPyaxTaEoILp1N803Q3Ziga1vTNu3k96AX0qjIfo/+OdC6IdKcdGt/3PbLMuy5x/lffdlxLKR6HeRCNwzSlCe4Yc2kv9waBw4w09ku/x